# Analiza Danych Facebook z Apify

Ten notebook pozwala na analizę danych pobieranych z konta Facebook za pomocą Apify API.

## Co można sprawdzić:
1. **Surowe dane z Apify** - bezpośrednio z ostatniego run'a
2. **Dane po parsowaniu** - jak są przetwarzane przed zapisem do bazy
3. **Dane w bazie danych** - co ostatecznie trafia do PostgreSQL
4. **Statystyki i wizualizacje** - podstawowa analiza danych


In [1]:
# Importy i setup
import sys
import os
from pathlib import Path

# Dodaj backend/src do ścieżki
backend_src = Path(os.getcwd()).parent / 'src'
sys.path.insert(0, str(backend_src))

import asyncio
import httpx
import json
import pandas as pd
from datetime import datetime
from pprint import pprint

# Import konfiguracji
from config import settings

print("✅ Setup zakończony")
print(f"Backend src path: {backend_src}")

✅ Setup zakończony
Backend src path: /Users/lukaszmarchlewicz/Desktop/Centrum_Operacyjne_Mieszkańca/backend/src


## 1. Konfiguracja

Podaj parametry dla Apify:

In [4]:
# Konfiguracja Apify
# APIFY_API_KEY = settings.APIFY_API_KEY
ACTOR_ID = "apify~facebook-posts-scraper"  # lub 'apify/facebook-posts-scraper'

# Sprawdź czy mamy klucz API
if not APIFY_API_KEY:
    print("❌ BŁĄD: Brak APIFY_API_KEY w .env")
    print("Dodaj do backend/.env:")
    print("APIFY_API_KEY=apify_api_***********")
else:
    print(f"✅ APIFY_API_KEY załadowany: {APIFY_API_KEY[:20]}...")

✅ APIFY_API_KEY załadowany: apify_api_HAKbMQQzgQ...


## 2. Pobierz listę ostatnich run'ów z Apify

Sprawdźmy jakie run'y Facebook scrapera są dostępne:

In [6]:
async def get_recent_runs(actor_id: str, api_key: str, limit: int = 10):
    """Pobierz listę ostatnich run'ów dla danego actora"""
    url = f"https://api.apify.com/v2/acts/{actor_id}/runs"
    params = {
        "token": api_key,
        "limit": limit,
        "desc": 1  # Od najnowszych
    }
    
    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        return response.json()

# Pobierz listę run'ów
runs_data = await get_recent_runs(ACTOR_ID, APIFY_API_KEY, limit=10)
runs = runs_data.get('data', {}).get('items', [])

print(f"\n📊 Znaleziono {len(runs)} ostatnich run'ów:\n")

for i, run in enumerate(runs[:5], 1):
    run_id = run['id']
    status = run['status']
    started_at = run.get('startedAt', 'N/A')
    finished_at = run.get('finishedAt', 'N/A')
    stats = run.get('stats', {})
    
    print(f"{i}. Run ID: {run_id}")
    print(f"   Status: {status}")
    print(f"   Started: {started_at}")
    print(f"   Finished: {finished_at}")
    print(f"   Stats: {stats}")
    print()


📊 Znaleziono 10 ostatnich run'ów:

1. Run ID: qp9dyI9PG045lkE2U
   Status: SUCCEEDED
   Started: 2026-01-11T20:57:19.913Z
   Finished: 2026-01-11T20:57:24.683Z
   Stats: {}

2. Run ID: WNj5BeQqPoikOVd7B
   Status: SUCCEEDED
   Started: 2026-01-11T20:56:49.641Z
   Finished: 2026-01-11T20:57:12.903Z
   Stats: {}

3. Run ID: m41QRjF3RZZRLOHLU
   Status: SUCCEEDED
   Started: 2026-01-11T20:56:19.738Z
   Finished: 2026-01-11T20:56:45.513Z
   Stats: {}

4. Run ID: bdfM0kWmQfrbWpfdc
   Status: SUCCEEDED
   Started: 2026-01-11T20:55:13.777Z
   Finished: 2026-01-11T20:55:19.108Z
   Stats: {}

5. Run ID: NbD5S6kMfLO8rQAiR
   Status: SUCCEEDED
   Started: 2026-01-11T20:54:38.164Z
   Finished: 2026-01-11T20:55:07.534Z
   Stats: {}



## 3. Pobierz dane z ostatniego успешного run'a

Wybierzmy ostatni SUCCEEDED run i pobierzmy z niego dane:

In [7]:
async def get_run_dataset(run_id: str, api_key: str):
    """Pobierz dataset z konkretnego run'a"""
    url = f"https://api.apify.com/v2/actor-runs/{run_id}/dataset/items"
    params = {"token": api_key}
    
    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        return response.json()

# Znajdź ostatni SUCCEEDED run
succeeded_run = None
for run in runs:
    if run['status'] == 'SUCCEEDED':
        succeeded_run = run
        break

if not succeeded_run:
    print("❌ Brak zakończonych pomyślnie run'ów")
    print("Uruchom najpierw scraper Facebook lub poczekaj na zakończenie aktualnego run'a")
else:
    run_id = succeeded_run['id']
    print(f"✅ Wybrany run: {run_id}")
    print(f"   Started: {succeeded_run.get('startedAt')}")
    print(f"   Finished: {succeeded_run.get('finishedAt')}\n")
    
    # Pobierz dane
    raw_data = await get_run_dataset(run_id, APIFY_API_KEY)
    
    print(f"📦 Pobrano {len(raw_data)} postów z Apify\n")
    
    # Zapisz do zmiennej globalnej
    facebook_posts_raw = raw_data

✅ Wybrany run: qp9dyI9PG045lkE2U
   Started: 2026-01-11T20:57:19.913Z
   Finished: 2026-01-11T20:57:24.683Z

📦 Pobrano 1 postów z Apify



In [8]:
raw_data = await get_run_dataset("m41QRjF3RZZRLOHLU", APIFY_API_KEY)
facebook_posts_raw = raw_data

In [9]:
len(facebook_posts_raw)

20

In [25]:
for row in facebook_posts_raw:
        print("-"*10)
        print(row['text'])

----------
SiS Rybno i Rumian w dzisiejszej kolejce w Lidze Halowej w Byszwałdzie o Puchar Wójta Gminy Lubawa strzelili w 4 spotkaniach aż 38 bramki, tracąc zaledwie 2 gole ‼️ 

Rumian – Torpeda Rożental 6:1
Rumian – Kajman Lubawa 11:0
SIS Rybno – P&Ł Transport 6:1
Kajman Lubawa – SIS Rybno 0:15
----------
Sportowe wyzwania 2025 dobiegły końca. Czekamy na zgłoszenia i wyniki uczestników 🥳

Z końcem 2025 roku dobiegły finału trzy całoroczne rywalizacje sportowe organizowane na terenie gminy Rybno. Był to czas pełen aktywności, wytrwałości i sportowej pasji – zarówno dla miłośników dwóch kółek, biegania, jak i nordic walking. Teraz nadszedł moment podsumowań i weryfikacji osiągniętych wyników.

Uczestnicy mogli wziąć udział w jednej lub kilku z następujących rywalizacji:

- 1100 km rowerem po gminie Rybno w 2025 roku,
- 550 km biegiem po gminie Rybno w 2025 roku,
- 450 km Nordic Walking po gminie Rybno w 2025 roku.

Celem inicjatywy była promocja aktywnego i zdrowego stylu życia, popular

## 4. Analiza surowych danych z Apify

Zobaczmy jakie pola zwraca Apify Facebook scraper:

In [11]:
if 'facebook_posts_raw' in globals() and facebook_posts_raw:
    # Pierwszy post jako przykład
    print("📄 PRZYKŁADOWY POST (surowe dane z Apify):\n")
    print(json.dumps(facebook_posts_raw[0], indent=2, ensure_ascii=False))
    
    print("\n" + "="*80 + "\n")
    
    # Wszystkie klucze w pierwszym poście
    print("🔑 DOSTĘPNE POLA W DANYCH:\n")
    for key in facebook_posts_raw[0].keys():
        value = facebook_posts_raw[0][key]
        value_type = type(value).__name__
        
        # Skrócona wartość dla wyświetlenia
        if isinstance(value, str) and len(value) > 100:
            value_preview = value[:100] + "..."
        else:
            value_preview = value
        
        print(f"  • {key:20s} ({value_type:10s}): {value_preview}")
else:
    print("❌ Brak danych - uruchom najpierw poprzednią komórkę")

📄 PRZYKŁADOWY POST (surowe dane z Apify):

{
  "facebookUrl": "https://www.facebook.com/serwis.informacyjny.syla",
  "postId": "1284009377086775",
  "pageName": "serwis.informacyjny.syla",
  "url": "https://www.facebook.com/reel/1633036224738436/",
  "time": "2026-01-11T20:01:12.000Z",
  "timestamp": 1768161672,
  "user": {
    "id": "100064331756232",
    "name": "Serwis informacyjny - Syla",
    "profileUrl": "https://www.facebook.com/100064331756232",
    "profilePic": "https://scontent-hou1-1.xx.fbcdn.net/v/t39.30808-1/591745891_1253269593494087_7300312126123238740_n.jpg?stp=cp0_dst-jpg_s50x50_tt6&_nc_cat=103&ccb=1-7&_nc_sid=2d3e12&_nc_ohc=qXXYrtoPVkwQ7kNvwFGi4XL&_nc_oc=AdkTgwLy62GFcYPN4bVapp5XVtVQC5WmhgZa7PWtvhwrEkPLdWtAodgPDS857AMuxhE&_nc_zt=24&_nc_ht=scontent-hou1-1.xx&_nc_gid=KNPtI98OZegsor8lXr4PMw&oh=00_AfoWveq_vG6XPaU_OM4uU8v4xMC36o3lEv8qQIBGdJKJ0w&oe=6969DB7F"
  },
  "text": "SiS Rybno i Rumian w dzisiejszej kolejce w Lidze Halowej w Byszwałdzie o Puchar Wójta Gminy Lubawa s

## 5. Symulacja parsowania danych

Zobaczmy jak scraper przetwarza dane przed zapisem do bazy:

In [12]:
def parse_facebook_post(post: dict) -> dict:
    """Symulacja parsowania jak w apify_facebook.py"""
    
    # Post ID
    post_id = post.get('postId') or post.get('id')
    
    # Tekst posta
    text = post.get('text') or post.get('message') or post.get('content')
    
    # Tytuł - pierwsze 100 znaków
    title = text[:100] if text else "Brak tytułu"
    if '. ' in title:
        title = title.split('. ')[0] + '.'
    if text and len(text) > 100:
        title += '...'
    
    # URL posta
    post_url = post.get('url') or post.get('postUrl') or f"https://facebook.com/{post_id}"
    
    # Obrazek
    image_url = post.get('imageUrl') or post.get('image') or post.get('picture')
    
    # Data publikacji
    published_at = None
    timestamp = post.get('timestamp') or post.get('time') or post.get('createdTime')
    if timestamp:
        try:
            if isinstance(timestamp, (int, float)):
                published_at = datetime.fromtimestamp(timestamp)
            elif isinstance(timestamp, str):
                published_at = datetime.fromisoformat(timestamp.replace('Z', '+00:00'))
        except:
            pass
    
    # Statystyki
    likes = post.get('likes', 0)
    comments = post.get('comments', 0)
    shares = post.get('shares', 0)
    
    # Content z statystykami
    content = text or ""
    if likes or comments or shares:
        content += f"\n\n[{likes} polubień, {comments} komentarzy, {shares} udostępnień]"
    
    return {
        'title': title,
        'url': post_url,
        'content': content,
        'image_url': image_url,
        'external_id': f"fb_{post_id}",
        'author': 'Facebook',
        'published_at': published_at.isoformat() if published_at else None,
        'likes': likes,
        'comments': comments,
        'shares': shares
    }

if 'facebook_posts_raw' in globals() and facebook_posts_raw:
    # Parsuj wszystkie posty
    parsed_posts = [parse_facebook_post(post) for post in facebook_posts_raw]
    
    print(f"✅ Sparsowano {len(parsed_posts)} postów\n")
    
    print("📄 PRZYKŁADOWY POST (po parsowaniu):\n")
    print(json.dumps(parsed_posts[0], indent=2, ensure_ascii=False))
else:
    print("❌ Brak danych do parsowania")

✅ Sparsowano 20 postów

📄 PRZYKŁADOWY POST (po parsowaniu):

{
  "title": "SiS Rybno i Rumian w dzisiejszej kolejce w Lidze Halowej w Byszwałdzie o Puchar Wójta Gminy Lubawa s...",
  "url": "https://www.facebook.com/reel/1633036224738436/",
  "content": "SiS Rybno i Rumian w dzisiejszej kolejce w Lidze Halowej w Byszwałdzie o Puchar Wójta Gminy Lubawa strzelili w 4 spotkaniach aż 38 bramki, tracąc zaledwie 2 gole ‼️ \n\nRumian – Torpeda Rożental 6:1\nRumian – Kajman Lubawa 11:0\nSIS Rybno – P&Ł Transport 6:1\nKajman Lubawa – SIS Rybno 0:15\n\n[7 polubień, 0 komentarzy, 1 udostępnień]",
  "image_url": null,
  "external_id": "fb_1284009377086775",
  "author": "Facebook",
  "published_at": "2026-01-11T21:01:12",
  "likes": 7,
  "comments": 0,
  "shares": 1
}


## 6. Porównanie: Surowe vs Parsed

Zobaczmy różnicę między danymi surowymi a przetworzonymi:

In [13]:
if 'facebook_posts_raw' in globals() and 'parsed_posts' in globals():
    print("🔄 PORÓWNANIE DANYCH:\n")
    print("=" * 80)
    
    raw_post = facebook_posts_raw[0]
    parsed_post = parsed_posts[0]
    
    print("\n📦 SUROWE DANE (z Apify):")
    print(f"  Liczba pól: {len(raw_post)}")
    print(f"  Pola: {', '.join(raw_post.keys())}")
    
    print("\n🔧 PRZETWORZONE DANE (do bazy):")
    print(f"  Liczba pól: {len(parsed_post)}")
    print(f"  Pola: {', '.join(parsed_post.keys())}")
    
    print("\n📊 MAPOWANIE:\n")
    mapping = [
        ("postId/id", "external_id", f"fb_{raw_post.get('postId') or raw_post.get('id')}"),
        ("text/message", "title + content", parsed_post['title'][:50] + "..."),
        ("url/postUrl", "url", parsed_post['url']),
        ("imageUrl/image", "image_url", parsed_post['image_url']),
        ("timestamp/time", "published_at", parsed_post['published_at']),
        ("likes/comments/shares", "content (stats)", f"{parsed_post['likes']} / {parsed_post['comments']} / {parsed_post['shares']}"),
    ]
    
    for apify_field, db_field, value in mapping:
        print(f"  {apify_field:25s} → {db_field:20s} = {value}")
else:
    print("❌ Brak danych do porównania")

🔄 PORÓWNANIE DANYCH:


📦 SUROWE DANE (z Apify):
  Liczba pól: 21
  Pola: facebookUrl, postId, pageName, url, time, timestamp, user, text, likes, shares, topReactionsCount, isVideo, viewsCount, media, feedbackId, reactionLikeCount, reactionLoveCount, topLevelUrl, facebookId, pageAdLibrary, inputUrl

🔧 PRZETWORZONE DANE (do bazy):
  Liczba pól: 10
  Pola: title, url, content, image_url, external_id, author, published_at, likes, comments, shares

📊 MAPOWANIE:

  postId/id                 → external_id          = fb_1284009377086775
  text/message              → title + content      = SiS Rybno i Rumian w dzisiejszej kolejce w Lidze H...
  url/postUrl               → url                  = https://www.facebook.com/reel/1633036224738436/
  imageUrl/image            → image_url            = None
  timestamp/time            → published_at         = 2026-01-11T21:01:12
  likes/comments/shares     → content (stats)      = 7 / 0 / 1


## 7. Analiza DataFrame

Stwórzmy DataFrame dla lepszej wizualizacji:

In [14]:
if 'parsed_posts' in globals():
    # Utwórz DataFrame
    df = pd.DataFrame(parsed_posts)
    
    print("📊 STATYSTYKI POSTÓW:\n")
    print(f"Liczba postów: {len(df)}")
    print(f"\nKolumny: {list(df.columns)}")
    
    print("\n📈 STATYSTYKI ENGAGEMENT:\n")
    print(df[['likes', 'comments', 'shares']].describe())
    
    print("\n🏆 TOP 5 POSTÓW (według polubień):\n")
    top_posts = df.nlargest(5, 'likes')[['title', 'likes', 'comments', 'shares']]
    print(top_posts.to_string(index=False))
    
    # Wyświetl cały DataFrame
    print("\n📋 WSZYSTKIE POSTY:\n")
    display(df[['title', 'published_at', 'likes', 'comments', 'shares']].head(10))
else:
    print("❌ Brak danych do analizy")

📊 STATYSTYKI POSTÓW:

Liczba postów: 20

Kolumny: ['title', 'url', 'content', 'image_url', 'external_id', 'author', 'published_at', 'likes', 'comments', 'shares']

📈 STATYSTYKI ENGAGEMENT:

            likes   comments     shares
count   20.000000  20.000000  20.000000
mean    30.350000   2.950000   3.600000
std     37.155824   6.660765   4.794734
min      3.000000   0.000000   0.000000
25%      6.750000   0.000000   1.000000
50%     12.000000   0.000000   1.000000
75%     36.250000   1.500000   4.250000
max    136.000000  27.000000  15.000000

🏆 TOP 5 POSTÓW (według polubień):

                                                                                                     title  likes  comments  shares
 ❤️ WOŚP w Rybnie – gramy razem mimo wszystko! ❤️\n\nChoć w tym roku finał Wielka Orkiestra Świątecznej...    136         3      12
   Z wielką przyjemnością uczestniczyliśmy dziś w KARNA WOW Celebration – wyjątkowym wydarzeniu podsumo...     96         8       4
GŁOSOWANIE CZAS ST

,title,published_at,likes,comments,shares
0,SiS Rybno i Rumian w dzisiejszej kolejce w Lid...,2026-01-11T21:01:12,7,0,1
1,Sportowe wyzwania 2025 dobiegły końca....,2026-01-11T15:45:41,3,0,1
2,🙏 Dziękujemy organizatorom za to piękne i ważn...,2026-01-11T13:01:53,22,1,0
3,📣 PILNIE POSZUKUJĘ MIESZKANIA – DZIAŁDOWO 📣\n\...,2026-01-11T12:00:04,6,1,1
4,Fresh Flow Dance Academy - w tej szkole tańczą...,2026-01-11T11:00:08,28,1,1
5,✅ Rumian i SiS Rybno zagrają po dwa mecze w pr...,2026-01-11T10:00:08,4,0,1
6,Dzień dobry wszystkim! 🎄Mamy 11 stycznia 💙 Życ...,2026-01-11T09:01:26,58,14,0
7,"Mamy już kilka propozycji na licytacje, które ...",2026-01-11T07:38:05,10,0,1
8,"Przypominamy, że głosowanie potrwa dziś do godz.",2026-01-11T06:53:41,7,0,1
9,GŁOSOWANIE CZAS START ❄️📸 KONKURS FOTOGRAFICZN...,2026-01-10T22:13:18,76,4,15


## 8. Sprawdź dane w bazie danych

Połączmy się z bazą i zobaczmy co zostało zapisane:

In [17]:
from sqlmodel import Session, select
from database_helper import Source, Article, get_database_connection

# Połącz z bazą
print("🔌 Łączenie z bazą danych...\\n")

try:
    engine = get_database_connection(settings.DATABASE_URL)
    
    with Session(engine) as session:
        # Znajdź źródło Facebook
        statement = select(Source).where(Source.type == "social_media")
        fb_sources = session.exec(statement).all()
        
        print("📱 ŹRÓDŁA FACEBOOK W BAZIE:\\n")
        for source in fb_sources:
            print(f"  • {source.name} (ID: {source.id})")
            print(f"    URL: {source.url}")
            print(f"    Last scraped: {source.last_scraped}")
            print(f"    Config: {source.scraping_config}")
            print()
        
        if fb_sources:
            # Pobierz artykuły z pierwszego źródła
            source_id = fb_sources[1].id
            
            statement = select(Article).where(
                Article.source_id == source_id
            ).order_by(Article.scraped_at.desc()).limit(10)
            
            articles = session.exec(statement).all()
            
            print(f"\\n📰 OSTATNIE {len(articles)} ARTYKUŁÓW Z BAZY:\\n")
            
            for article in articles:
                print(f"  • {article.title[:60]}...")
                print(f"    ID: {article.id} | External ID: {article.external_id}")
                print(f"    Published: {article.published_at}")
                print(f"    Scraped: {article.scraped_at}")
                print(f"    Category: {article.category} | Tags: {article.tags}")
                print(f"    Processed: {article.processed}")
                print()
            
            # Stwórz DataFrame z bazy
            articles_df = pd.DataFrame([
                {
                    'id': a.id,
                    'title': a.title,
                    'external_id': a.external_id,
                    'published_at': a.published_at,
                    'scraped_at': a.scraped_at,
                    'category': a.category,
                    'processed': a.processed
                }
                for a in articles
            ])
            
            print("\\n📊 PODSUMOWANIE ARTYKUŁÓW W BAZIE:\\n")
            display(articles_df)
        else:
            print("\\n❌ Brak źródeł Facebook w bazie")
            print("Dodaj źródło przez API lub bezpośrednio do bazy danych")
            
except Exception as e:
    print(f"\\n❌ BŁĄD połączenia z bazą: {e}")
    print(f"\\n💡 Sprawdź:")
    print(f"  1. Czy PostgreSQL działa")
    print(f"  2. Czy DATABASE_URL w .env jest poprawny")
    print(f"  3. DATABASE_URL: {settings.DATABASE_URL[:50]}...")

🔌 Łączenie z bazą danych...\n
📱 ŹRÓDŁA FACEBOOK W BAZIE:\n
  • Gmina Działdowo Facebook (ID: 4)
    URL: https://facebook.com/GminaDzialdowo
    Last scraped: None
    Config: {'actor_id': 'apify/facebook-pages-scraper', 'max_posts': 20, 'apify_api_key': 'UZUPEŁNIJ_W_BAZIE', 'scraper_class': 'ApifyFacebookScraper', 'facebook_page_url': 'https://facebook.com/GminaDzialdowo'}

  • Facebook - Syla (ID: 5)
    URL: https://www.facebook.com/serwis.informacyjny.syla
    Last scraped: 2026-01-11 20:56:48.946608
    Config: {'actor_id': 'apify~facebook-posts-scraper', 'caption_text': False, 'apify_api_key': 'APIFY_KEY_REMOVED', 'results_limit': 20, 'scraper_class': 'ApifyFacebookScraper', 'facebook_page_url': 'https://www.facebook.com/serwis.informacyjny.syla'}

  • Facebook - Gmina Działdowo (ID: 6)
    URL: https://www.facebook.com/GminaDzialdowo
    Last scraped: 2026-01-11 20:57:18.958832
    Config: {'actor_id': 'apify~facebook-posts-scraper', 'caption_text': False, 'apify_api_key': 'APIF

,id,title,external_id,published_at,scraped_at,category,processed
0,387,✅ Rumian i SiS Rybno zagrają po dwa mecze w pr...,fb_1282372280583818,2026-01-09 15:01:04,2026-01-11 20:56:48.944328,Kultura,True
1,386,❤️ WOŚP w Rybnie – gramy razem mimo wszystko! ...,fb_1282434477244265,2026-01-09 16:35:12,2026-01-11 20:56:48.941520,Kultura,True
2,385,Polecamy nową stronę przyrodniczą z terenu Gmi...,fb_1282561813898198,2026-01-09 20:10:14,2026-01-11 20:56:48.939003,Rekreacja,True
3,384,Hallo Rybno! Czy ktoś może znalazł tablice rej...,fb_1282604477227265,2026-01-09 21:43:58,2026-01-11 20:56:48.936401,Urząd,True
4,383,Dzień dobry wszystkim! 🎄Mamy 10 stycznia 💙 Życ...,fb_1282833827204330,2026-01-10 06:11:03,2026-01-11 20:56:48.933764,Kultura,True
5,382,Aktualne oferty pracy w Powiatowym Urzędzie Pr...,fb_1282912210529825,2026-01-10 09:01:39,2026-01-11 20:56:48.930353,Biznes,True
6,381,❤️ WOŚP w Rybnie – gramy razem mimo wszystko! ...,fb_1282958110525235,2026-01-10 10:35:06,2026-01-11 20:56:48.926734,Kultura,True
7,380,❤️ WOŚP w Rybnie – gramy razem mimo wszystko! ...,fb_1282964963857883,2026-01-10 10:49:15,2026-01-11 20:56:48.923710,Kultura,True
8,379,"‼️ PRACA Sala w Siwym ""Siwy Dym"" ‼️\nStanowisk...",fb_1283060107181702,2026-01-10 13:45:20,2026-01-11 20:56:48.920849,Biznes,True
9,378,Z wielką przyjemnością uczestniczyliśmy dziś w...,fb_1283322817155431,2026-01-10 21:39:20,2026-01-11 20:56:48.917530,Kultura,True


In [19]:
if 'parsed_posts' in globals() and 'articles_df' in globals():
    print("🔍 PORÓWNANIE: APIFY vs BAZA DANYCH\n")
    print("=" * 80)
    
    # External IDs z Apify
    apify_ids = set([p['external_id'] for p in parsed_posts])
    
    # External IDs z bazy
    db_ids = set(articles_df['external_id'].tolist())
    
    print(f"\n📦 Posty w Apify: {len(apify_ids)}")
    print(f"💾 Posty w bazie: {len(db_ids)}")
    
    # Posty w Apify, których nie ma w bazie
    missing_in_db = apify_ids - db_ids
    if missing_in_db:
        print(f"\n⚠️  Posty w Apify, których BRAK w bazie: {len(missing_in_db)}")
        for ext_id in list(missing_in_db)[:5]:
            print(f"  • {ext_id}")
    else:
        print("\n✅ Wszystkie posty z Apify są w bazie!")
    
    # Posty w bazie, których nie ma w Apify (starsze)
    only_in_db = db_ids - apify_ids
    if only_in_db:
        print(f"\n📚 Posty tylko w bazie (starsze): {len(only_in_db)}")
        for ext_id in list(only_in_db)[:5]:
            print(f"  • {ext_id}")
else:
    print("❌ Brak danych do porównania")

🔍 PORÓWNANIE: APIFY vs BAZA DANYCH


📦 Posty w Apify: 20
💾 Posty w bazie: 10

⚠️  Posty w Apify, których BRAK w bazie: 10
  • fb_1283671020453944
  • fb_1283828737104839
  • fb_1283640207123692
  • fb_1283731817114531
  • fb_1283613827126330


## 9. Porównanie: Apify vs Baza Danych

Sprawdźmy czy wszystkie posty z Apify trafiły do bazy:

## 10. Export danych

Zapisz dane do plików CSV dla dalszej analizy:

In [ ]:
if 'df' in globals():
    # Export parsed posts
    output_dir = Path(os.getcwd()) / 'exports'
    output_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Apify data
    apify_file = output_dir / f'facebook_apify_{timestamp}.csv'
    df.to_csv(apify_file, index=False, encoding='utf-8')
    print(f"✅ Zapisano dane Apify do: {apify_file}")
    
    # Database data
    if 'articles_df' in globals():
        db_file = output_dir / f'facebook_database_{timestamp}.csv'
        articles_df.to_csv(db_file, index=False, encoding='utf-8')
        print(f"✅ Zapisano dane z bazy do: {db_file}")
    
    # Raw JSON
    if 'facebook_posts_raw' in globals():
        json_file = output_dir / f'facebook_raw_{timestamp}.json'
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(facebook_posts_raw, f, indent=2, ensure_ascii=False)
        print(f"✅ Zapisano surowe dane JSON do: {json_file}")
    
    print(f"\n📁 Wszystkie pliki w: {output_dir}")
else:
    print("❌ Brak danych do exportu")

In [34]:
list_missing = list(missing_in_db)

In [43]:
df[df['external_id'].isin(list_missing)]

,title,url,content,image_url,external_id,author,published_at,likes,comments,shares
0,SiS Rybno i Rumian w dzisiejszej kolejce w Lid...,https://www.facebook.com/reel/1633036224738436/,SiS Rybno i Rumian w dzisiejszej kolejce w Lid...,None,fb_1284009377086775,Facebook,2026-01-11T21:01:12,7,0,1
1,Sportowe wyzwania 2025 dobiegły końca....,https://www.facebook.com/serwis.informacyjny.s...,Sportowe wyzwania 2025 dobiegły końca. Czekamy...,None,fb_1283828737104839,Facebook,2026-01-11T15:45:41,3,0,1
2,🙏 Dziękujemy organizatorom za to piękne i ważn...,https://www.facebook.com/serwis.informacyjny.s...,🙏 Dziękujemy organizatorom za to piękne i ważn...,None,fb_1283731817114531,Facebook,2026-01-11T13:01:53,22,1,0
3,📣 PILNIE POSZUKUJĘ MIESZKANIA – DZIAŁDOWO 📣\n\...,https://www.facebook.com/serwis.informacyjny.s...,📣 PILNIE POSZUKUJĘ MIESZKANIA – DZIAŁDOWO 📣\n\...,None,fb_1283700537117659,Facebook,2026-01-11T12:00:04,6,1,1
4,Fresh Flow Dance Academy - w tej szkole tańczą...,https://www.facebook.com/serwis.informacyjny.s...,Fresh Flow Dance Academy - w tej szkole tańczą...,None,fb_1283671020453944,Facebook,2026-01-11T11:00:08,28,1,1
5,✅ Rumian i SiS Rybno zagrają po dwa mecze w pr...,https://www.facebook.com/serwis.informacyjny.s...,✅ Rumian i SiS Rybno zagrają po dwa mecze w pr...,None,fb_1283640207123692,Facebook,2026-01-11T10:00:08,4,0,1
6,Dzień dobry wszystkim! 🎄Mamy 11 stycznia 💙 Życ...,https://www.facebook.com/reel/1216950423734088/,Dzień dobry wszystkim! 🎄Mamy 11 stycznia 💙 Życ...,None,fb_1283613827126330,Facebook,2026-01-11T09:01:26,58,14,0
7,"Mamy już kilka propozycji na licytacje, które ...",https://www.facebook.com/serwis.informacyjny.s...,"Mamy już kilka propozycji na licytacje, które ...",None,fb_1283578483796531,Facebook,2026-01-11T07:38:05,10,0,1
8,"Przypominamy, że głosowanie potrwa dziś do godz.",https://www.facebook.com/serwis.informacyjny.s...,"Przypominamy, że głosowanie potrwa dziś do god...",None,fb_1283558780465168,Facebook,2026-01-11T06:53:41,7,0,1
9,GŁOSOWANIE CZAS START ❄️📸 KONKURS FOTOGRAFICZN...,https://www.facebook.com/serwis.informacyjny.s...,GŁOSOWANIE CZAS START ❄️📸 KONKURS FOTOGRAFICZN...,None,fb_1283337273820652,Facebook,2026-01-10T22:13:18,76,4,15


## 🎯 Podsumowanie

Ten notebook pozwolił Ci:

1. ✅ Pobrać ostatnie dane z Apify
2. ✅ Przeanalizować strukturę surowych danych
3. ✅ Zobaczyć jak dane są parsowane
4. ✅ Sprawdzić co trafia do bazy danych
5. ✅ Porównać dane z różnych źródeł
6. ✅ Wyeksportować dane do CSV/JSON

### 🔧 Możliwe usprawnienia:

- Dodaj wizualizacje (matplotlib/plotly)
- Analiza sentymentu tekstu
- Wykrywanie trendów w popularności postów
- Automatyczna analiza hashtagów
